# 06 — FAN-DLinear on ETTh1

This notebook implements **Frequency-Adaptive Normalization (FAN)** around a separate DLinear backbone and evaluates the same task as our original experiment: use 336 hourly observations of all seven ETTh1 features to forecast the next 96 hours. The original `DLinear` implementation remains unchanged.

FAN removes the top-K Fourier components independently for every sample and channel. DLinear forecasts the residual, while a small shared MLP forecasts how the removed components evolve. Training uses the paper's two-part objective: overall forecast MSE plus an equally weighted MSE for the future dominant-frequency component.

## 1. Fair protocol and the one necessary adaptation

The FAN paper used lookback 96, horizon 96, and `K=4` for ETTh1. Our established DLinear protocol uses lookback 336. We therefore compare `K ∈ {4, 8, 12}` using **mean validation MSE across seeds 2021–2023**. After freezing K, the test split is evaluated once. Everything else matches the original DLinear experiment: chronological 6/2/2 ETT split, training-only z-score fitting, moving average 25, shared DLinear projections, Adam, learning rate 0.005 with the type-1 schedule, batch size 32, ten epochs, and patience three.

This notebook deliberately reports the selected method even if it fails to improve the baseline.

In [ ]:
from pathlib import Path
import copy
import json
import statistics
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader

from ts_project.data import FEATURE_COLUMNS, build_window_datasets, prepare_etth1
from ts_project.models import FANDLinear, dominant_frequency_component
from ts_project.training import seed_everything

plt.style.use('seaborn-v0_8-whitegrid')
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG = {
    'input_length': 336, 'prediction_length': 96, 'channels': 7,
    'moving_average': 25, 'top_k_candidates': [4, 8, 12],
    'prior_weight': 1.0, 'seeds': [2021, 2022, 2023],
    'batch_size': 32, 'learning_rate': 0.005,
    'max_epochs': 10, 'patience': 3,
}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
CONFIG

## 2. Prepare the identical leakage-safe ETTh1 windows

In [ ]:
data = prepare_etth1(PROJECT_ROOT / 'data' / 'raw' / 'ETTh1.csv')
datasets = build_window_datasets(
    data, input_length=CONFIG['input_length'],
    prediction_length=CONFIG['prediction_length'],
)
pd.DataFrame({
    'windows': {name: len(dataset) for name, dataset in datasets.items()},
    'first_origin': {name: dataset.first_origin for name, dataset in datasets.items()},
    'last_origin': {name: dataset.last_origin for name, dataset in datasets.items()},
})

## 3. Audit the FAN decomposition

The FFT operation must be lossless before forecasting: residual plus selected dominant component must reconstruct the input. The test below also shows how much input energy each K removes on one training window.

In [ ]:
sample_x, _ = datasets['train'][0]
audit_rows = []
for top_k in CONFIG['top_k_candidates']:
    residual, dominant = dominant_frequency_component(sample_x.unsqueeze(0), top_k)
    reconstruction_error = (residual + dominant - sample_x.unsqueeze(0)).abs().max().item()
    removed_energy = dominant.square().mean().item() / sample_x.square().mean().item()
    audit_rows.append({'K': top_k, 'max reconstruction error': reconstruction_error,
                       'dominant/input energy': removed_energy})
pd.DataFrame(audit_rows).set_index('K')

## 4. Training and evaluation helpers

Model selection sees only validation MSE. Per-feature MSE and MAE are calculated over every scaled forecast value, exactly like the existing project reports.

In [ ]:
def make_loaders(seed):
    generator = torch.Generator().manual_seed(seed)
    return {
        'train': DataLoader(datasets['train'], batch_size=CONFIG['batch_size'],
                            shuffle=True, generator=generator),
        'validation': DataLoader(datasets['validation'], batch_size=CONFIG['batch_size']),
        'test': DataLoader(datasets['test'], batch_size=CONFIG['batch_size']),
    }

@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    squared = torch.zeros(len(FEATURE_COLUMNS), dtype=torch.float64)
    absolute = torch.zeros(len(FEATURE_COLUMNS), dtype=torch.float64)
    count = 0
    for inputs, targets in loader:
        errors = model(inputs.to(device)).cpu() - targets
        squared += (errors ** 2).sum(dim=(0, 1), dtype=torch.float64)
        absolute += errors.abs().sum(dim=(0, 1), dtype=torch.float64)
        count += targets.shape[0] * targets.shape[1]
    per_mse = squared / count
    per_mae = absolute / count
    return {
        'MSE': per_mse.mean().item(), 'MAE': per_mae.mean().item(),
        'per_feature': {feature: {'MSE': per_mse[i].item(), 'MAE': per_mae[i].item()}
                        for i, feature in enumerate(FEATURE_COLUMNS)},
    }

def train_fan(top_k, seed):
    seed_everything(seed)
    loaders = make_loaders(seed)
    model = FANDLinear(
        input_length=CONFIG['input_length'], prediction_length=CONFIG['prediction_length'],
        channels=CONFIG['channels'], top_k=top_k,
        moving_average=CONFIG['moving_average'],
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
    best_state, best_validation, best_epoch = None, float('inf'), 0
    stale = 0
    history = []
    for epoch in range(1, CONFIG['max_epochs'] + 1):
        model.train()
        total_forecast, total_prior, total_count = 0.0, 0.0, 0
        current_lr = optimizer.param_groups[0]['lr']
        for inputs, targets in loaders['train']:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss, parts = model.training_loss(
                inputs, targets, prior_weight=CONFIG['prior_weight'])
            loss.backward()
            optimizer.step()
            elements = targets.numel()
            total_forecast += parts['forecast_mse'].item() * elements
            total_prior += parts['dominant_mse'].item() * elements
            total_count += elements
        validation = evaluate(model, loaders['validation'])['MSE']
        history.append({
            'K': top_k, 'seed': seed, 'epoch': epoch, 'learning_rate': current_lr,
            'train_forecast_MSE': total_forecast / total_count,
            'train_dominant_MSE': total_prior / total_count,
            'validation_MSE': validation,
        })
        if validation < best_validation:
            best_validation, best_epoch = validation, epoch
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
            if stale >= CONFIG['patience']:
                break
        next_lr = CONFIG['learning_rate'] * (0.5 ** (epoch - 1))
        for group in optimizer.param_groups:
            group['lr'] = next_lr
    model.load_state_dict(best_state)
    return model, {
        'K': top_k, 'seed': seed, 'best_epoch': best_epoch,
        'validation_MSE': best_validation,
        'parameters': sum(p.numel() for p in model.parameters()),
        'history': history,
    }

## 5. Select K on validation only

The following cell trains all nine development runs. It intentionally does not touch the test loader.

In [ ]:
started = time.perf_counter()
development_models = {}
development_records = []
for top_k in CONFIG['top_k_candidates']:
    for seed in CONFIG['seeds']:
        model, record = train_fan(top_k, seed)
        development_models[(top_k, seed)] = model.cpu()
        development_records.append(record)
        print(f"K={top_k:2d}, seed={seed}: val MSE={record['validation_MSE']:.6f} "
              f"at epoch {record['best_epoch']}")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
selection_table = (pd.DataFrame(development_records)
                   .groupby('K')['validation_MSE']
                   .agg(['mean', 'std']).sort_values('mean'))
selected_k = int(selection_table.index[0])
print(f'Frozen K={selected_k}; selection took {(time.perf_counter()-started)/60:.2f} minutes')
display(selection_table)

## 6. Open the test split once for the frozen K

Only the three already validation-selected checkpoints for the winning K are evaluated below.

In [ ]:
test_records = []
for seed in CONFIG['seeds']:
    model = development_models[(selected_k, seed)].to(device)
    metrics = evaluate(model, make_loaders(seed)['test'])
    test_records.append({'seed': seed, **metrics})
    print(f"seed={seed}: test MSE={metrics['MSE']:.6f}, MAE={metrics['MAE']:.6f}")
fan_mse = statistics.mean(row['MSE'] for row in test_records)
fan_mae = statistics.mean(row['MAE'] for row in test_records)
fan_mse_std = statistics.stdev(row['MSE'] for row in test_records)
fan_mae_std = statistics.stdev(row['MAE'] for row in test_records)
{'K': selected_k, 'MSE_mean': fan_mse, 'MSE_std': fan_mse_std,
 'MAE_mean': fan_mae, 'MAE_std': fan_mae_std}

## 7. Comparison with the original DLinear

The primary baseline is the project's three-seed DLinear mean under the identical protocol. The paper-reported DLinear number is included only as an external reference. Positive improvement percentages mean lower error.

In [ ]:
baseline_summary = pd.read_csv(
    PROJECT_ROOT / 'results' / 'dlinear' / 'daily_adaptation_sequence' / 'summary.csv')
baseline = baseline_summary.query("horizon == 96 and model == 'DLinear'").iloc[0]
overall_comparison = pd.DataFrame([
    {'model': 'Original DLinear (local, 3 seeds)',
     'MSE': baseline.test_MSE_mean, 'MSE std': baseline.test_MSE_std,
     'MAE': baseline.test_MAE_mean, 'MAE std': baseline.test_MAE_std},
    {'model': f'FAN-DLinear (K={selected_k}, 3 seeds)',
     'MSE': fan_mse, 'MSE std': fan_mse_std, 'MAE': fan_mae, 'MAE std': fan_mae_std},
    {'model': 'Paper-reported original DLinear',
     'MSE': 0.375, 'MSE std': float('nan'), 'MAE': 0.399, 'MAE std': float('nan')},
]).set_index('model')
mse_improvement = 100 * (baseline.test_MSE_mean - fan_mse) / baseline.test_MSE_mean
mae_improvement = 100 * (baseline.test_MAE_mean - fan_mae) / baseline.test_MAE_mean
display(overall_comparison)
print(f'FAN-DLinear vs original DLinear: MSE improvement {mse_improvement:+.2f}%, '
      f'MAE improvement {mae_improvement:+.2f}%')

In [ ]:
baseline_features = pd.read_csv(
    PROJECT_ROOT / 'results' / 'dlinear' / 'daily_adaptation_sequence' / 'per_feature_h096.csv'
).query("model == 'DLinear'").set_index('feature')
feature_rows = []
for feature in FEATURE_COLUMNS:
    fan_feature_mse = statistics.mean(r['per_feature'][feature]['MSE'] for r in test_records)
    fan_feature_mae = statistics.mean(r['per_feature'][feature]['MAE'] for r in test_records)
    base_mse = baseline_features.loc[feature, 'test_MSE_mean']
    base_mae = baseline_features.loc[feature, 'test_MAE_mean']
    feature_rows.append({
        'feature': feature, 'DLinear MSE': base_mse, 'FAN MSE': fan_feature_mse,
        'MSE improvement %': 100 * (base_mse - fan_feature_mse) / base_mse,
        'DLinear MAE': base_mae, 'FAN MAE': fan_feature_mae,
        'MAE improvement %': 100 * (base_mae - fan_feature_mae) / base_mae,
    })
feature_comparison = pd.DataFrame(feature_rows).set_index('feature')
display(feature_comparison)

In [ ]:
ax = feature_comparison[['DLinear MSE', 'FAN MSE']].plot.bar(figsize=(11, 4.5))
ax.set(title=f'ETTh1 336→96 per-feature MSE (FAN K={selected_k})', ylabel='MSE')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## Recorded result from the completed CPU run

The predeclared three-seed validation grid selected **K=8**. Mean validation MSE was 0.736238 for K=8, 0.738323 for K=12, and 0.745578 for K=4. All three were worse than the original DLinear validation mean of 0.641923.

| Model | Test MSE (mean ± sample std) | Test MAE (mean ± sample std) |
|---|---:|---:|
| Original DLinear | 0.381133 ± 0.012862 | 0.405212 ± 0.015507 |
| FAN-DLinear, K=8 | 0.424488 ± 0.007979 | 0.436627 ± 0.003621 |

FAN-DLinear was **11.38% worse in MSE** and **7.75% worse in MAE**. It improved only OT (17.45% MSE and 7.55% MAE) and degraded all six load channels. The validation failure and test failure agree, so this configuration should not replace the original model. The outcome also reinforces the earlier finding that normalization-like transformations help OT but are not uniformly beneficial across ETTh1 features.

## 8. Save compact reproducibility artifacts

In [ ]:
output = PROJECT_ROOT / 'results' / 'dlinear' / 'fan' / 'etth1' / 'horizon_096'
output.mkdir(parents=True, exist_ok=True)
selection_table.to_csv(output / 'selection_by_k.csv')
pd.DataFrame([item for record in development_records for item in record['history']]).to_csv(
    output / 'training_history.csv', index=False)
overall_comparison.to_csv(output / 'overall_comparison.csv')
feature_comparison.to_csv(output / 'per_feature_comparison.csv')
summary = {
    'selected_K': selected_k, 'selection': selection_table.reset_index().to_dict('records'),
    'FAN-DLinear': {'MSE_mean': fan_mse, 'MSE_std': fan_mse_std,
                    'MAE_mean': fan_mae, 'MAE_std': fan_mae_std},
    'Original DLinear': {'MSE_mean': float(baseline.test_MSE_mean),
                         'MSE_std': float(baseline.test_MSE_std),
                         'MAE_mean': float(baseline.test_MAE_mean),
                         'MAE_std': float(baseline.test_MAE_std)},
    'improvement_percent': {'MSE': mse_improvement, 'MAE': mae_improvement},
    'config': CONFIG,
}
with (output / 'metrics.json').open('w') as file:
    json.dump(summary, file, indent=2)
print('Saved:', output.relative_to(PROJECT_ROOT))

## 9. Interpretation

Use the validation table to judge whether the K choice is stable, the overall table to answer whether FAN beat original DLinear, and the feature table to see whether the aggregate result hides opposing channel-level effects. A test improvement without a credible validation advantage should be treated cautiously rather than as evidence for further tuning on the test set.